In [ ]:
# === Setup ===
# Runtime: <1m with OAI_FAST_MODE=1
# Hardware: CPU smoke
# Network: none
# Competition-safe: Yes for the declared profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# 1. Linear Regression (From Scratch)

Trong notebook này, chúng ta sẽ tự xây dựng thuật toán Linear Regression chỉ bằng `NumPy`. KHÔNG dùng scikit-learn.

**Mục tiêu:**
1. Implement hàm forward pass (dự đoán).
2. Implement hàm Loss (MSE).
3. Implement quá trình tính Gradient và cập nhật trọng số (Gradient Descent).
4. Đóng gói thành class `LinearRegression`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Tạo dữ liệu giả định
X = 2 * np.random.rand(100, 1)
y = 4 + 3 * X + np.random.randn(100, 1) # y = 3x + 4 + noise

plt.scatter(X, y)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Dữ liệu huấn luyện')
plt.show()

## Bước 1: Khởi tạo mô hình
Mô hình $y = XW + b$.

In [ ]:
class LinearRegressionFromScratch:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iters = n_iterations
        self.W = None
        self.b = None
        self.loss_history = []
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # Khởi tạo tham số ngẫu nhiên
        self.W = np.random.randn(n_features, 1)
        self.b = np.random.randn(1)
        
        # Gradient Descent
        for i in range(self.n_iters):
            # Forward pass: dự đoán y_pred
            y_pred = np.dot(X, self.W) + self.b
            
            # Tính Loss (MSE)
            loss = (1/n_samples) * np.sum((y_pred - y)**2)
            self.loss_history.append(loss)
            
            # Tính Gradients
            # dL/dW = (2/N) * X^T * (y_pred - y)
            dW = (2/n_samples) * np.dot(X.T, (y_pred - y))
            # dL/db = (2/N) * sum(y_pred - y)
            db = (2/n_samples) * np.sum(y_pred - y)
            
            # Cập nhật tham số
            self.W -= self.lr * dW
            self.b -= self.lr * db
            
            if (i+1) % 100 == 0:
                print(f'Iteration {i+1}: Loss = {loss:.4f}')
                
    def predict(self, X):
        return np.dot(X, self.W) + self.b

In [ ]:
# Chạy thử mô hình
model = LinearRegressionFromScratch(learning_rate=0.1, n_iterations=1000)
model.fit(X, y)

print(f'\nTrọng số học được: W = {model.W[0][0]:.4f}, b = {model.b[0]:.4f}')
print(f'Trọng số thực tế: W = 3, b = 4')
assert model.loss_history[-1] < model.loss_history[0]
assert model.predict(np.zeros((2, 1))).shape == (2, 1)
# Gradient check cho một trọng số trên batch nhỏ
eps = 1e-6
w0 = model.W.copy(); residual = X @ w0 + model.b - y
analytic = float((2 / len(X)) * (X.T @ residual)[0, 0])
plus = np.mean((X @ (w0 + eps) + model.b - y) ** 2)
minus = np.mean((X @ (w0 - eps) + model.b - y) ** 2)
numeric = (plus - minus) / (2 * eps)
assert np.isclose(analytic, numeric, rtol=1e-4, atol=1e-6)

## Visualize Loss và Đường Hồi Quy

In [ ]:
plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(model.loss_history)
plt.title('Loss History')
plt.xlabel('Iteration')
plt.ylabel('MSE')

# Plot Regression Line
plt.subplot(1, 2, 2)
plt.scatter(X, y, color='blue', label='Data')
plt.plot(X, model.predict(X), color='red', label='Prediction')
plt.title('Regression Line')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()

plt.tight_layout()
plt.show()